# Non-Degenerate Perturbation Theory: a Gaussian Bump in an Infinite Well

**Phase 3** of `Perturbation_and_Basis_Methods_Plan.md`. Same physical idea as
the legacy `1D Pertubaton Theory.ipynb` (a localized bump perturbing an
infinite-square-well eigenstate) but fixed (that notebook referenced its
perturbation array before defining it, so it only ran if cells were executed
out of order) and, unlike the legacy notebook, actually validated: 1st- and
2nd-order Rayleigh-Schrodinger perturbation theory
(`perturbation.first_order_energy`, `second_order_energy`, `first_order_state`)
is checked against direct diagonalization of the truncated `H0+H'` matrix
(`perturbation.exact_diagonalization`) across a sweep of perturbation
strengths -- confirming both that perturbation theory's error shrinks at the
*correct power* of the strength as it gets small, and that it visibly breaks
down once the strength is no longer small (shown explicitly, not hidden).

In [1]:
import sys
sys.path.insert(0, r".")
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

import basis as bs
import perturbation as pt

MEDIA_DIR = Path('media')
MEDIA_DIR.mkdir(exist_ok=True)


## Setup

Infinite square well, `L=5`, 40-mode box basis. Perturbation: a Gaussian bump
`V'(x) = V0 * exp(-(x-L/2)^2/(2*w^2))` centered in the well (`w=0.2`), an
antinode for the odd-n states -- state `n=3` is used throughout. `V0` is swept
from deep in the perturbative regime up to strong coupling.

In [2]:
L, N, num_modes = 5.0, 2000, 40
x = np.linspace(0, L, N)
box = bs.box_basis(x, L, num_modes)

x_c, w = L / 2, 0.2
bump_shape = np.exp(-(x - x_c) ** 2 / (2 * w ** 2))
H_prime_unit = pt.matrix_elements(box, bump_shape)  # H' for V0=1; scale by V0 below

n_index = 2  # n=3 (0-indexed into box.eigenfunctions/energies)
E0 = box.energies[n_index]
print(f"unperturbed E_3 = {E0:.5f}")


unperturbed E_3 = 1.77653


## Energy correction vs. exact diagonalization

For each `V0`, compare the exact diagonalized energy (tracking whichever exact
eigenstate has the largest overlap with the unperturbed state, via
`perturbation.track_state` -- robust to any incidental level reordering) to
the 1st-order (`E0+V0*E1`) and 1st+2nd-order (`+V0^2*E2`) perturbative
predictions.

In [3]:
V0_list = np.array([0.001, 0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0, 3.0])
E1_unit = pt.first_order_energy(H_prime_unit)[n_index]
E2_unit = pt.second_order_energy(H_prime_unit, box.energies, n_index)

E_exact_list, E_pt1_list, E_pt2_list = [], [], []
for V0 in V0_list:
    H_prime = V0 * H_prime_unit
    evals, evecs = pt.exact_diagonalization(box.energies, H_prime, lam=1.0)
    idx = pt.track_state(evecs, n_index)
    E_exact_list.append(evals[idx])
    E_pt1_list.append(E0 + V0 * E1_unit)
    E_pt2_list.append(E0 + V0 * E1_unit + V0 ** 2 * E2_unit)

E_exact_arr = np.array(E_exact_list)
E_pt1_arr = np.array(E_pt1_list)
E_pt2_arr = np.array(E_pt2_list)
err1 = np.abs(E_pt1_arr - E_exact_arr)
err2 = np.abs(E_pt2_arr - E_exact_arr)

print(f"{'V0':>8} {'E_exact':>12} {'E_PT1':>12} {'E_PT2':>12} {'err_PT1':>10} {'err_PT2':>10}")
for V0, Ee, E1v, E2v, e1, e2 in zip(V0_list, E_exact_arr, E_pt1_arr, E_pt2_arr, err1, err2):
    print(f"{V0:8.3f} {Ee:12.6f} {E1v:12.6f} {E2v:12.6f} {e1:10.2e} {e2:10.2e}")


      V0      E_exact        E_PT1        E_PT2    err_PT1    err_PT2
   0.001     1.776705     1.776705     1.776705   1.04e-08   2.13e-12
   0.010     1.778287     1.778286     1.778287   1.04e-06   2.14e-09
   0.050     1.785341     1.785315     1.785341   2.57e-05   2.68e-07
   0.100     1.794203     1.794101     1.794205   1.02e-04   2.16e-06
   0.300     1.830120     1.829246     1.830179   8.74e-04   5.93e-05
   0.500     1.866705     1.864390     1.866984   2.31e-03   2.79e-04
   1.000     1.960328     1.952252     1.962626   8.08e-03   2.30e-03
   2.000     2.150713     2.127975     2.169470   2.27e-02   1.88e-02
   3.000     2.334962     2.303698     2.397063   3.13e-02   6.21e-02


In [4]:
results = []
def check(name, cond, detail=""):
    results.append((name, bool(cond), detail))
    print(f"{'PASS' if cond else 'FAIL'}: {name}  {detail}")


In [5]:
# --- Check A: error scaling in the small-V0 regime ---
small = V0_list <= 0.1
slope1 = np.polyfit(np.log(V0_list[small]), np.log(err1[small]), 1)[0]
slope2 = np.polyfit(np.log(V0_list[small]), np.log(err2[small]), 1)[0]
print(f"1st-order energy error scales as V0^{slope1:.3f} (theory: 2)")
print(f"2nd-order energy error scales as V0^{slope2:.3f} (theory: 3)")
check("A 1st-order PT energy error scales as V0^2", 1.8 < slope1 < 2.2, f"slope={slope1:.3f}")
check("A 2nd-order PT energy error scales as V0^3", 2.8 < slope2 < 3.2, f"slope={slope2:.3f}")


1st-order energy error scales as V0^1.996 (theory: 2)
2nd-order energy error scales as V0^3.002 (theory: 3)
PASS: A 1st-order PT energy error scales as V0^2  slope=1.996
PASS: A 2nd-order PT energy error scales as V0^3  slope=3.002


In [6]:
# --- Check B: 1st-order wavefunction correction matches the exact
# eigenvector's shape to O(V0^2), for a weak perturbation ---
wf_errs = []
for V0 in (0.01, 0.1, 0.3):
    H_prime = V0 * H_prime_unit
    c1 = pt.first_order_state(H_prime, box.energies, n_index)  # already the full (scaled) correction
    e_n = np.zeros(num_modes)
    e_n[n_index] = 1.0
    c_total = e_n + c1
    c_total /= np.linalg.norm(c_total)

    evals, evecs = pt.exact_diagonalization(box.energies, H_prime, lam=1.0)
    idx = pt.track_state(evecs, n_index)
    c_exact = evecs[:, idx].copy()
    if c_exact[n_index] < 0:
        c_exact = -c_exact
    if c_total[n_index] < 0:
        c_total = -c_total
    wf_errs.append(np.max(np.abs(c_total - c_exact)))
    print(f"V0={V0}: max coefficient error = {wf_errs[-1]:.3e}")

check("B wavefunction correction error shrinks as V0 decreases",
      wf_errs[0] < wf_errs[1] < wf_errs[2], f"{[f'{e:.2e}' for e in wf_errs]}")
check("B wavefunction correction is accurate for weak perturbation", wf_errs[0] < 1e-5, f"{wf_errs[0]:.2e}")


V0=0.01: max coefficient error = 5.822e-07
V0=0.1: max coefficient error = 5.958e-05
V0=0.3: max coefficient error = 5.616e-04
PASS: B wavefunction correction error shrinks as V0 decreases  ['5.82e-07', '5.96e-05', '5.62e-04']
PASS: B wavefunction correction is accurate for weak perturbation  5.82e-07


In [7]:
# --- Check C: perturbation theory visibly breaks down at strong coupling
# -- 2nd order should NOT uniformly improve on 1st order once V0 is no
# longer small; shown explicitly rather than only checking the regime
# where PT is expected to work ---
check("C 2nd-order PT is worse than 1st-order at strong coupling (V0=3) -- expected breakdown",
      err2[-1] > err1[-1], f"err_PT1={err1[-1]:.2e}  err_PT2={err2[-1]:.2e}  at V0={V0_list[-1]}")


PASS: C 2nd-order PT is worse than 1st-order at strong coupling (V0=3) -- expected breakdown  err_PT1=3.13e-02  err_PT2=6.21e-02  at V0=3.0


In [8]:
n_pass = sum(1 for _, ok, _ in results if ok)
print(f"\n{n_pass}/{len(results)} Phase 3 checks passed")
assert n_pass == len(results), "Phase 3 validation failed"



5/5 Phase 3 checks passed


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(V0_list, E_exact_arr, 'o-', label='exact diagonalization', color='k')
axes[0].plot(V0_list, E_pt1_arr, 's--', label='1st-order PT', color='C0')
axes[0].plot(V0_list, E_pt2_arr, '^--', label='1st+2nd-order PT', color='C3')
axes[0].set_xlabel('V0'); axes[0].set_ylabel('E_3(V0)')
axes[0].set_title('Perturbed energy vs. bump strength')
axes[0].legend()

axes[1].loglog(V0_list, err1, 's-', label='1st-order error', color='C0')
axes[1].loglog(V0_list, err2, '^-', label='1st+2nd-order error', color='C3')
axes[1].loglog(V0_list, 0.5 * V0_list ** 2, ':', color='C0', label='~V0^2')
axes[1].loglog(V0_list, 0.05 * V0_list ** 3, ':', color='C3', label='~V0^3')
axes[1].set_xlabel('V0'); axes[1].set_ylabel('|E_PT - E_exact|')
axes[1].set_title('PT error scaling (and breakdown at large V0)')
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(MEDIA_DIR / 'perturbation_theory_convergence.png', dpi=150)
plt.show()


C:\Users\Hasan's Laptop\AppData\Local\Temp\ipykernel_22304\2037067882.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
